## What Is a Context Manager?

- an object that __manages a resource or some temporary state__
- e.g.,
  - opening and closing files
  - acquiring and releasing locks
  - opening and closing network connections
  - temporarily changing settings
- a context manager is useful when you need a clear **start** and **finish**:
  - acquire something
  - use it
  - clean it up (which should happen even if an error occurs)

## The __`with`__ Statement
* the most common way to use a context manager is with the __`with`__ statement
* the classic example is opening a file, which you are probably already familiar with

In [ ]:
with open('example.txt', 'w') as outfile:
    print('Hello, context managers!', file=outfile)

* when the __`with`__ block ends, Python automatically closes the file
* as a result we get
  - less boilerplate
  - safer cleanup
  - clearer code
* without the __`with`__, you might write something like this:

In [ ]:
outfile = open('example.txt', 'w')

try:
    print('Hello, context managers!', file=outfile)
finally:
    file.close()

* these two versions express a similar idea, but the __`with`__ version is cleaner and easier to read
* the __`with`__ statement is often a cleaner alternative to __`try` / `finally`__ when setup and cleanup naturally go together


## A First Look at Resource Management
* another example using a file

In [ ]:
with open('example.txt') as file:
    # is the file open at this point? what is this? what are we printing?
    print(f'in with block, file.closed = {file.closed}')
    
    for line in file:
        print(line, end='') 

print(f'outside of with block, file.closed = {file.closed}')

* inside the __`with`__ block, the file is open and available
* once the block ends, the file is closed automatically


## The Protocol Behind Context Managers

* a context manager works by defining two special methods:

  - __`__enter__`__
  - __`__exit__`__


* these methods tell Python what to do:
  - when entering the __`with`__ block
  - when leaving the __`with`__ block


In [ ]:
class BasicManager:
    """A basic resource manager. Doesn't do anything, but useful as an example."""
    
    def __enter__(self):
        """Return a 'resource'"""
        print('Entering context')
        
        return 'resource'

    def __exit__(self, exc_type, exc_value, traceback):
        """Arguments to this method discussed below."""
        print('Exiting context')

In [ ]:
with BasicManager() as resource:
    print('Inside block:', resource)

### What happened?

- __`__enter__`__ ran first
- its return value was assigned to __`resource`__
- the block executed
- __`__exit__`__ ran when the block finished

* (that happens whether the block ends normally or because of an exception)

##  __`__enter__`__ and __`__exit__`__ in More Detail

* a custom context manager can manage setup and cleanup explicitly

In [ ]:
class ManagedList:
    def __enter__(self):
        """Create a "managed list" upon entry."""
        print('Creating list')
        self.data = []
        return self.data

    def __exit__(self, exc_type, exc_value, traceback):
        """Empty the list upon exit."""
        print('Cleaning up list')
        self.data.clear()

In [ ]:
with ManagedList() as data:
    data.append(10)
    data.append(20)
    data.append(30)
    data.remove(20)
    print(data)

Here:

- __`__enter__`__ creates and returns the managed object
- __`__exit__`__ performs cleanup

The three arguments to __`__exit__`__:

- __`exc_type`__
- __`exc_value`__
- __`traceback`__

contain exception information if an error occurred inside the block

In [ ]:
class VerboseManager:
    def __enter__(self):
        print('Entering')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print('Exiting')
        print('exc_type:', exc_type)
        print('exc_value:', exc_value)

In [ ]:
try:
    with VerboseManager():
        print('About to divide by zero')
        1 / 0
except ZeroDivisionError:
    print('Caught ZeroDivisionError outside the with block')

* notice that __`__exit__`__ still runs even though an exception occurred
* by default, the exception still propagates after __`__exit__`__ finishes


## Suppressing Exceptions

A context manager can choose to suppress an exception by returning __`True`__ from __`__exit__`__


In [ ]:
class SuppressZeroDivision:
    def __enter__(self):
        print('Entering')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print('Exiting')
        return exc_type is ZeroDivisionError

In [ ]:
with SuppressZeroDivision():
    print('Before error')
    1 / 0
    print('This line will not run')

print('Program continues')

* this pattern should be used carefully, but it helps illustrate the role of __`__exit__`__

## __`contextlib.closing`__

* sometimes you have an object with a __`.close()`__ method, but it is not itself a context manager
* __`contextlib.closing`__ wraps such an object so it can be used with __`with`__


In [ ]:
from contextlib import closing

class FakeConnection:
    def __init__(self):
        self.closed = False

    def send(self, message):
        if self.closed:
            raise RuntimeError('Connection is closed')
        print('Sending:', message)

    def close(self):
        self.closed = True
        print('Connection closed')

with closing(FakeConnection()) as connection:
    connection.send('hello')
    connection.send('world')

## __`contextlib.contextmanager`__
* sometimes defining a whole class just to support __`with`__ feels like too much
* __`contextlib.contextmanager`__ allows you to build a context manager from a generator function

In [ ]:
from contextlib import contextmanager

@contextmanager
def simple_manager():
    print('Setup')
    try:
        yield 'resource'
    finally:
        print('Cleanup')

In [ ]:
with simple_manager() as value:
    print('Inside block:', value)

## How it works
* with __`@contextmanager`__:
  - code before __`yield`__ acts like __`__enter__`__
  - the yielded value becomes the __`as`__ value
  - code after __`yield`__ acts like __`__exit__`__

* this is often a convenient way to create basic context managers

## A More Practical __`@contextmanager`__ Example
* a small timer context manager...

In [1]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label):
    start = time.time() # this is the equivalent of __enter__
    try:
        yield # this becomes the 'as' value (None in this case)
    finally: # equivalent of __exit__
        end = time.time()
        print(f'{label} took {end - start:.4f} seconds')

In [2]:
with timer('sleeping') as thing:
    print('thing is', thing)
    time.sleep(1.4)

sleeping took 0.0000 seconds


NameError: name 'thing' is not defined

In [11]:
import random

random_nums = [random.randint(1, 100) for _ in range(50_000_000)]

with timer('sorting'):
    random_nums.sort()

5.435791969299316


In [18]:
random_nums = [random.randint(1, 100) for _ in range(50_000_000)]
%time random_nums.sort()

CPU times: user 5.27 s, sys: 239 ms, total: 5.51 s
Wall time: 5.5 s


## Exercise
* Create a context manager with __`@contextmanager`__ that suppresses exceptions
  * allow the caller to enter multiple exceptions, comma-separated, if desired
* Create a context manager with __`@contextmanager`__ that simulates a lock
  * upon entry the lock should be set
  * upon exit it should be released
  * if the lock is set upon entry an exception should be raised